# Regular initial condition series solutions

This notebook derives the early radiation-era series solutions for the regular scalar
perturbation modes (adiabatic and isocurvature), reproducing the
*Regular initial condition series solutions* section of the
[CAMB notes](https://cosmologist.info/notes/CAMB.pdf).
These series set the scalar initial conditions in CAMB (`subroutine initial` in
`fortran/equations.f90`). Everything is derived with sympy using the covariant scalar
equations defined in
`camb.symbolic` (see the [ScalEqs notebook](https://camb.readthedocs.io/en/latest/ScalEqs.html)).

Assumptions (as in the notes):

* massless free-streaming neutrinos; photons and baryons in lowest-order tight coupling
  ($\pi_\gamma = 0$, $v_b = 3q_\gamma/4$), cold baryons ($p_b = c_{s,b}^2 = 0$);
* CDM frame ($A = v_c = 0$), the frame used by CAMB;
* general curvature, entering through $\beta_2 \equiv {\rm Kf}_1 = 1 - 3K/k^2$
  (and ${\rm Kf}_\ell = 1 - \ell(\ell+2)K/k^2$);
* series in conformal time $\tau$ about $\tau = 0$; the displayed results keep the orders
  quoted in the notes (up to $O(\tau^3)$).

Variables follow `camb.symbolic` conventions: $\Delta_i$ are fractional density
perturbations, $q_i$ heat fluxes ($v_i = q_i$ for pressureless matter), $\pi_r$ the
massless-neutrino anisotropic stress and $G_3$, $G_4$, $G_5$ the higher neutrino multipoles,
$\eta$ the three-curvature perturbation. The notebook takes a few minutes to run.

In [1]:
import sympy
from IPython.display import Markdown, display
from sympy import Eq, Function, Rational, Symbol, diff, symbols
from sympy.polys.fields import field as frac_field

from camb import symbolic as cs
from camb.symbolic import (
    Delta_b,
    Delta_c,
    Delta_g,
    Delta_r,
    G_eq,
    H,
    K,
    K_fac,
    Kf,
    P,
    Pi,
    a,
    cdm_subs,
    csq_b,
    define_variable,
    delta_eqs,
    drag_t,
    eta,
    k,
    kappa,
    p_b,
    phi,
    pi_g,
    pi_r,
    q,
    q_g,
    q_r,
    rho,
    rho_b,
    rho_c,
    rho_de,
    rho_g,
    rho_nu,
    rho_r,
    sigma,
    subs,
    t,
    tot_pert_subs,
    tot_subs,
    v_b,
    v_c,
    var_subs,
    vel_eqs,
    z,
)

sympy.init_printing()

# display helpers: show conformal time as tau and K_fac as beta_2, output the
# math as markdown (auto-sized boxes), and order series with the lowest power first
tau_d, beta2_d = symbols("tau beta_2")


def disp(expr):
    return sympy.S(expr).subs(K_fac, beta2_d).subs(t, tau_d)


def dmath(obj):
    display(Markdown("$\\displaystyle " + (obj if isinstance(obj, str) else sympy.latex(disp(obj))) + "$"))


def laurent_dict(expr):
    # {order: coefficient} for a finite Laurent expression in t
    # (expr.coeff misses negative powers when sympy merges denominators)
    d = {}
    for term in sympy.Add.make_args(sympy.expand(expr)):
        num, den = sympy.fraction(sympy.together(term))
        degs_n = {m[0] for m in sympy.Poly(num, t).monoms()}
        degs_d = {m[0] for m in sympy.Poly(den, t).monoms()}
        assert len(degs_n) == 1 and len(degs_d) == 1, term
        n = degs_n.pop() - degs_d.pop()
        d[n] = d.get(n, 0) + sympy.cancel(term / t**n)
    return {n: c for n, c in d.items() if c != 0}


def series_latex(expr, orders=None):
    # latex of a series in t, in increasing powers
    ld = laurent_dict(expr)
    out = ""
    for n in orders if orders is not None else sorted(ld):
        if n not in ld:
            continue
        s = sympy.latex(disp(sympy.factor(ld[n]) * t**n))
        if not out:
            out = s
        else:
            out += " - " + s[1:] if s.startswith("-") else " + " + s
    return out or "0"

## Background evolution

Following the notes define $\omega \equiv \Omega_m H_0/\sqrt{\Omega_R}$ with
$\Omega_R = \Omega_\gamma + \Omega_\nu$, and the radiation fractions
$R_\nu = \Omega_\nu/\Omega_R$, $R_c = \Omega_c/\Omega_m$ (so $R_\gamma = 1-R_\nu$,
$R_b = 1-R_c$). The perturbation series depend on the background only through $\omega$
(and $K$), so without loss of generality we can set $\kappa = H_0 = \Omega_m = 1$.

We solve the Friedmann equation $S'^2 = \kappa S^4\rho/3 - K S^2$ order by order in
$\tau$ with $S(0)=0$. A cosmological constant $\Omega_v$ only enters at $O(\tau^5)$
(as the result shows), so it does not affect the perturbation series below — nor would
early quintessence, which is why the modes are unchanged when adding a quintessence field.

In [2]:
omega, omv, Rv, Rc = symbols("omega Omega_v R_nu R_c", positive=True)
Rb = 1 - Rc  # baryon fraction of the matter
Rg = 1 - Rv  # photon fraction of the radiation

omR = 1 / omega**2  # Omega_R, with Omega_m = H0 = 1
Ord = 3  # keep displayed terms up to tau^Ord
M = Ord + 4  # order of the series ansatz

Scoeffs = [Symbol(f"c{i}") for i in range(2, M + 2)]
S_ansatz = 1 / omega**2 * (omega * t + sum(c * t**i for i, c in enumerate(Scoeffs, 2)))
rho_bg = 3 * (omR / a**4 + 1 / a**3 + omv)  # radiation + matter + Lambda
friedmann = sympy.expand(diff(a, t) ** 2 - (a**4 * rho_bg / 3 - K * a**2))  # cancel powers of a first
friedmann = sympy.expand(friedmann.subs(a, S_ansatz).doit())
sol_bg = {}
for n in range(1, M + 1):
    eqn = sympy.expand(friedmann.coeff(t, n).subs(sol_bg))
    if eqn == 0:
        continue
    c_n = sorted([c for c in Scoeffs if eqn.has(c)], key=str)[0]
    sol_bg[c_n] = sympy.simplify(sympy.solve(eqn, c_n)[0])
    sol_bg = {key: v.subs(sol_bg) for key, v in sol_bg.items()}

S_full = sympy.expand(S_ansatz.subs(sol_bg))

# check against the series quoted in the notes
S_notes = (omega * t + (omega * t) ** 2 / 4 - K / 6 * omega * t**3 - K / 48 * omega**2 * t**4) / omega**2
assert all(sympy.expand(S_full - S_notes).coeff(t, n) == 0 for n in range(5))

dmath(sympy.latex(disp(a)) + " = " + series_latex(S_full, orders=range(1, 6)) + r" + \mathcal{O}\left(\tau^{6}\right)")

$\displaystyle a{\left(\tau \right)} = \frac{\tau}{\omega} + \frac{\tau^{2}}{4} -  \frac{K \tau^{3}}{6 \omega} -  \frac{K \tau^{4}}{48} + \frac{\tau^{5} \left(K^{2} \omega^{2} + 12 \Omega_{v}\right)}{120 \omega^{3}} + \mathcal{O}\left(\tau^{6}\right)$

## Lowest-order tight coupling

At lowest order in the tight-coupling expansion $\pi_\gamma = 0$ and $v_b = 3q_\gamma/4$.
Eliminating $\dot{v}_b$ between the baryon and photon velocity equations gives the drag
term used in the notes,
$$ S n_e\sigma_T\left(\tfrac{4}{3}v_b - q_\gamma\right) =
 -\frac{\rho_b}{3\rho_b+4\rho_\gamma}\left(k\Delta_\gamma + 4\mathcal{H}v_b\right), $$
and with it a closed equation for $q_\gamma'$.

In [3]:
drag = Function("drag")(t)
vb_eq = [e for e in vel_eqs if e.lhs == diff(v_b, t)][0]
qg_eq = [e for e in vel_eqs if e.lhs == diff(q_g, t)][0]
tc_subs = [Eq(pi_g, 0), Eq(csq_b, 0), Eq(p_b, 0), Eq(v_b, 3 * q_g / 4)]
sys = [subs(cdm_subs, subs(tc_subs, e.subs(drag_t, drag))).doit() for e in [vb_eq, qg_eq]]
dqg_sym = Symbol("dqg")
sys = [e.subs(diff(q_g, t), dqg_sym) for e in sys]
sol_tc = sympy.solve([e.lhs - e.rhs for e in sys], [drag, dqg_sym])

drag_expected = -rho_b / (3 * rho_b + 4 * rho_g) * (k * Delta_g + 3 * H * q_g)
assert sympy.simplify(sol_tc[drag] - drag_expected) == 0
dmath(Eq(drag_t, drag_expected))

$\displaystyle \left(- q_{g}{\left(\tau \right)} + \frac{4 v_{b}{\left(\tau \right)}}{3}\right) \operatorname{opacity}{\left(\tau \right)} = - \frac{\left(k \Delta_{g}{\left(\tau \right)} + 3 H{\left(\tau \right)} q_{g}{\left(\tau \right)}\right) \rho_{b}{\left(\tau \right)}}{3 \rho_{b}{\left(\tau \right)} + 4 \rho_{g}{\left(\tau \right)}}$

## The closed equation set

In the CDM frame we evolve $\Delta_\gamma$, $\Delta_\nu$, $\Delta_b$, $\Delta_c$,
$q_\gamma$ (tightly coupled), $q_\nu$, the neutrino hierarchy $\pi_\nu$, $G_3$, $G_4$
(with $G_5$ starting at $O(\tau^4)$), and $\eta$. The metric quantities
$\dot{h}$, $z$, $\sigma$, $\phi$ are fixed by the constraint equations (`var_subs`), and
the totals $\delta$, $q$, $\Pi$ by the component sums (`tot_pert_subs`); massive
neutrinos and dark energy perturbations are not included ($\rho_\nu = \rho_{de} = 0$).

In [4]:
G_3 = define_variable("G_3")
G_4 = define_variable("G_4")
G_5 = define_variable("G_5")

eqs = [[e for e in delta_eqs if e.lhs == diff(var, t)][0] for var in [Delta_g, Delta_r, Delta_b, Delta_c]]
eqs.append(Eq(diff(q_g, t), sol_tc[dqg_sym]))
eqs.append([e for e in vel_eqs if e.lhs == diff(q_r, t)][0])
eqs += [G_eq(2), G_eq(3), G_eq(4)]
eqs.append([e for e in cs.pert_eqs if e.lhs == diff(eta, t)][0])

for e in eqs:
    dmath(subs(tc_subs, subs(cdm_subs, e)).doit())

$\displaystyle \frac{d}{d \tau} \Delta_{g}{\left(\tau \right)} = - k q_{g}{\left(\tau \right)} - 4 \dot{h}{\left(\tau \right)}$

$\displaystyle \frac{d}{d \tau} \Delta_{r}{\left(\tau \right)} = - k q_{r}{\left(\tau \right)} - 4 \dot{h}{\left(\tau \right)}$

$\displaystyle \frac{d}{d \tau} \Delta_{b}{\left(\tau \right)} = - \frac{3 k q_{g}{\left(\tau \right)}}{4} - 3 \dot{h}{\left(\tau \right)}$

$\displaystyle \frac{d}{d \tau} \Delta_{c}{\left(\tau \right)} = - 3 \dot{h}{\left(\tau \right)}$

$\displaystyle \frac{d}{d \tau} q_{g}{\left(\tau \right)} = \frac{4 k \Delta_{g}{\left(\tau \right)} \rho_{g}{\left(\tau \right)}}{9 \rho_{b}{\left(\tau \right)} + 12 \rho_{g}{\left(\tau \right)}} - \frac{9 H{\left(\tau \right)} q_{g}{\left(\tau \right)} \rho_{b}{\left(\tau \right)}}{9 \rho_{b}{\left(\tau \right)} + 12 \rho_{g}{\left(\tau \right)}}$

$\displaystyle \frac{d}{d \tau} q_{r}{\left(\tau \right)} = - \frac{2 \beta_{2} k \pi_{r}{\left(\tau \right)}}{3} + \frac{k \Delta_{r}{\left(\tau \right)}}{3}$

$\displaystyle \frac{d}{d \tau} \pi_{r}{\left(\tau \right)} = - \frac{k \left(3 G_{3}{\left(\tau \right)} {Kf}_{2} - 2 q_{r}{\left(\tau \right)}\right)}{5} + \frac{8 k \sigma{\left(\tau \right)}}{15}$

$\displaystyle \frac{d}{d \tau} G_{3}{\left(\tau \right)} = - \frac{k \left(4 G_{4}{\left(\tau \right)} {Kf}_{3} - 3 \pi_{r}{\left(\tau \right)}\right)}{7}$

$\displaystyle \frac{d}{d \tau} G_{4}{\left(\tau \right)} = - \frac{k \left(- 4 G_{3}{\left(\tau \right)} + 5 G_{5}{\left(\tau \right)} {Kf}_{4}\right)}{9}$

$\displaystyle \frac{d}{d \tau} \eta{\left(\tau \right)} = - \frac{2 K z{\left(\tau \right)} + \kappa a^{2}{\left(\tau \right)} q{\left(\tau \right)}}{k}$

In [5]:
zero_subs = [Eq(rho_nu, 0), Eq(rho_de, 0), Eq(cs.p_nu, 0)]

solve_vars = [Delta_g, Delta_r, Delta_b, Delta_c, q_g, q_r, pi_r, G_3, G_4, eta]
var_names = ["clxg", "clxr", "clxb", "clxc", "qg", "qr", "pir", "G3", "G4", "eta"]


def to_fluid_vars(expr):
    # reduce an expression to the evolved variables (+G_5) and background functions
    expr = subs(tc_subs, subs(cdm_subs, expr))
    expr = subs(var_subs, expr)
    expr = subs(tot_pert_subs, expr)
    expr = subs(tot_subs, expr)
    expr = subs(tc_subs, subs(cdm_subs, expr))
    expr = subs(zero_subs, expr)
    return expr.doit()


# Kf_l = 1 - K l(l+2)/k^2 in terms of beta_2 = Kf_1, using K = k^2(1-beta_2)/3
K_series_sub = Eq(K, k**2 * (1 - K_fac) / 3)
Kf_subs = [Eq(Kf[ell], 1 - Rational(ell * (ell + 2), 3) * (1 - K_fac)) for ell in (2, 3, 4)]

closed_eqs = [subs(Kf_subs, to_fluid_vars(e.lhs - e.rhs)) for e in eqs]

# explicit background functions of t (Lambda dropped: it enters only at O(tau^5))
S = sympy.expand(S_full.subs(omv, 0))
bg_subs = [
    Eq(a, S),
    Eq(H, sympy.cancel(diff(S, t) / S)),
    Eq(rho_g, 3 * (1 - Rv) * omR / kappa / a**4),
    Eq(rho_r, 3 * Rv * omR / kappa / a**4),
    Eq(rho_b, 3 * Rb / kappa / a**3),
    Eq(rho_c, 3 * Rc / kappa / a**3),
    Eq(rho_nu, 0),
    Eq(rho_de, 0),
    Eq(p_b, 0),
]

## Series ansatz and solution method

Each variable is expanded as $v(\tau) = \sum_i s_{v,i}\,\tau^i$; regularity requires the
$\ell$-th multipoles to vanish fast enough at $\tau\to0$ ($\pi_\nu$ starts at $O(\tau)$,
$G_3$ at $O(\tau^2)$, $G_4$ at $O(\tau^3)$, $G_5$ at $O(\tau^4)$).
Substituting the ansatz and the background series turns each equation
into a Laurent series in $\tau$ whose coefficients must all vanish. We repeatedly solve
the lowest-order nonvanishing coefficient of every
equation and substitute back; each pass moves one order higher.

Three tricks keep the computer algebra fast:

* the equations are linear and homogeneous in the perturbations, so every Laurent
  coefficient is a sparse linear form $\sum_i c_i\, s_i$ with $c_i$ in the rational
  function field $\mathbb{Q}(\omega, R_\nu, R_c, \beta_2)$, on which sympy's `FracField`
  arithmetic is fast and automatically normalized;
* the equations are dimensionally homogeneous — the coefficient of $\tau^n$ in any
  (dimensionless) variable is homogeneous of degree $n$ in $(k, \omega)$ — so we can set
  $k = 1$ while solving and restore the powers of $k$ at the end;
* Laurent expansion of the background rational functions is done by direct series
  division of numerator by denominator (no multivariate gcds).

The free parameters remaining at the end are the five seed values
$s_{\eta,0}$, $s_{\Delta_b,0}$, $s_{\Delta_c,0}$, $s_{\Delta_\nu,0}$, $s_{q_\nu,0}$
which parametrize the five regular modes.

In [6]:
coeffs = {}
series_map = {}
for var, name in zip(solve_vars, var_names):
    coeffs[name] = [Symbol(f"s_{name}_{i}") for i in range(M + 1)]
    series_map[var] = sum(c * t**i for i, c in enumerate(coeffs[name]))
sG5 = [Symbol("s_G5_4"), Symbol("s_G5_5")]
series_map[G_5] = sG5[0] * t**4 + sG5[1] * t**5

# regularity conditions on the neutrino multipoles
fixed_zero = {
    coeffs["pir"][0]: 0,
    coeffs["G3"][0]: 0,
    coeffs["G3"][1]: 0,
    coeffs["G4"][0]: 0,
    coeffs["G4"][1]: 0,
    coeffs["G4"][2]: 0,
}

seeds = [coeffs["eta"][0], coeffs["clxb"][0], coeffs["clxc"][0], coeffs["clxr"][0], coeffs["qr"][0]]

s_symbol_set = set(sG5)
for cvec in coeffs.values():
    s_symbol_set.update(cvec)
solvable = s_symbol_set - set(seeds) - set(fixed_zero)

NLOW = 4  # allow equation poles up to t^-NLOW
NMAX = M - 2  # highest equation order not affected by truncation of the ansatz

pert_terms = list(series_map.keys()) + [diff(v, t) for v in series_map]


def rational_laurent(f, nmax):
    # Laurent expansion of a rational function of t about t=0, up to t**nmax,
    # by direct series division (avoids slow multivariate gcds in cancel/series)
    num, den = sympy.fraction(sympy.together(f))
    cn = {deg[0]: c for deg, c in sympy.Poly(sympy.expand(num), t).as_dict().items()}
    cd = {deg[0]: c for deg, c in sympy.Poly(sympy.expand(den), t).as_dict().items()}
    if not cn:
        return sympy.S.Zero
    ln, ld = min(cn), min(cd)
    shift = ln - ld
    nterms = nmax - shift + 1
    if nterms <= 0:
        return sympy.S.Zero
    b = [1 / cd[ld]]  # series coefficients of t**ld/den
    for n in range(1, nterms):
        b.append(sympy.cancel(-sum(cd.get(ld + j, 0) * b[n - j] for j in range(1, n + 1)) / cd[ld]))
    res = sympy.S.Zero
    for m in range(nterms):
        c = sympy.cancel(sum(cn.get(ln + i, 0) * b[m - i] for i in range(m + 1)))
        if c != 0:
            res += c * t ** (shift + m)
    return res


def bg_laurent(f):
    f = subs(K_series_sub, subs(bg_subs, subs(bg_subs, f))).doit()
    return rational_laurent(f, NMAX + 1)


def laurent_coeffs(expr, nmax=NMAX, nlow=NLOW):
    # Laurent coefficients of an expression linear in the perturbations, after
    # substituting the background and perturbation series (and setting k=1)
    expr = sympy.expand(expr)
    parts = sympy.collect(expr, pert_terms, evaluate=False)
    total = sympy.S.Zero
    for pert, coeff in parts.items():
        if pert == 1:
            assert coeff == 0, coeff
            continue
        total += sympy.expand(bg_laurent(coeff) * pert.subs(series_map).doit().subs(fixed_zero))
    total = sympy.expand(total.subs(k, 1))
    return {n: total.coeff(t, n) for n in range(-nlow, nmax + 1) if total.coeff(t, n) != 0}

In [7]:
# Sparse linear forms {s_i: c_i} with c_i in the field QQ(omega, R_nu, R_c, beta_2)
FF, f_omega, f_Rv, f_Rc, f_b2 = frac_field([omega, Rv, Rc, K_fac], sympy.QQ)


def form_from_expr(expr):
    d = {}
    for term in sympy.Add.make_args(sympy.expand(expr)):
        ss = term.free_symbols & s_symbol_set
        assert len(ss) == 1, term  # linear and homogeneous in the perturbations
        key = ss.pop()
        d[key] = d.get(key, FF.zero) + FF.from_expr(term / key)
    return {key: v for key, v in d.items() if v}


def form_compose(form, solved):
    out = {}
    for key, c in form.items():
        if key in solved:
            for k2, c2 in solved[key].items():
                out[k2] = out.get(k2, FF.zero) + c * c2
        else:
            out[key] = out.get(key, FF.zero) + c
    return {key: v for key, v in out.items() if v}


def solve_forms(forms, unknowns):
    # sparse Gaussian elimination on linear forms = 0
    forms = [dict(f) for f in forms]
    solved = {}
    while True:
        cand = [(len([key for key in f if key in unknowns]), i) for i, f in enumerate(forms) if f]
        cand = [(nunk, i) for nunk, i in cand if nunk]
        if not cand:
            break
        _, i = min(cand)
        f = forms.pop(i)
        u = sorted([key for key in f if key in unknowns], key=str)[0]
        expr_form = {key: -v / f[u] for key, v in f.items() if key != u}
        solved = {key: form_compose(g, {u: expr_form}) for key, g in solved.items()}
        solved[u] = expr_form
        forms = [form_compose(g, {u: expr_form}) for g in forms if g]
    for g in forms:
        assert not g, ("inconsistent system", g)
    return solved


eq_forms = [{n: form_from_expr(c) for n, c in laurent_coeffs(e).items()} for e in closed_eqs]

# order-by-order solution of the series coefficients
sol_forms = {}
for it in range(30):
    lead_eqs = []
    orders = []
    for eqc in eq_forms:
        for n in sorted(eqc):
            eqc[n] = form_compose(eqc[n], sol_forms)
            if not eqc[n]:
                del eqc[n]
                continue
            if n <= NMAX - 1:
                lead_eqs.append(eqc[n])
                orders.append(n)
            break
    if not lead_eqs:
        break
    new = solve_forms(lead_eqs, solvable - set(sol_forms))
    assert new, (it, orders)
    sol_forms = {key: form_compose(f, new) for key, f in sol_forms.items()}
    sol_forms.update(new)
    print(f"pass {it}: leading equation orders {orders}, solved {len(new)} coefficients")

unresolved = solvable - set(sol_forms)
for key, f in sol_forms.items():
    if set(f) & unresolved:
        unresolved.add(key)

pass 0: leading equation orders [-1, -1, -1, -1, 0, 0, -2, 1, 2, -2], solved 6 coefficients
pass 1: leading equation orders [0, 0, 0, 0, 1, 1, 0, 2, 3, 0], solved 10 coefficients
pass 2: leading equation orders [1, 1, 1, 1, 2, 2, 1, 3, 4, 1], solved 10 coefficients


pass 3: leading equation orders [2, 2, 2, 2, 3, 3, 2, 4, 2], solved 9 coefficients


pass 4: leading equation orders [3, 3, 3, 3, 4, 4, 3, 3], solved 8 coefficients


pass 5: leading equation orders [4, 4, 4, 4, 4, 4], solved 6 coefficients


## The regular modes

Setting one seed to a nonzero value and the rest to zero gives the five regular modes.
For the adiabatic mode the seed $s_{\eta,0} = 2\beta_2$ corresponds to the normalization
$\chi_0 = -1$ used by CAMB, where $\chi$ is the comoving curvature perturbation.

We also evaluate auxiliary quantities: the constraint variables $\sigma$, $z$, the Weyl
potential $\phi$, the potentials
$\Phi = -\phi - \kappa S^2\Pi/(2k^2)$ and $\Psi = \phi - \kappa S^2\Pi/(2k^2)$,
$\chi = -\bar\eta/(2\beta_2)$ with $\bar\eta$ the comoving-frame value of $\eta$,
computed from the definition in the notes
$$ \bar\eta = 2\beta_2\left[\Phi +
\frac{2}{3}\Omega^{-1}\frac{\mathcal{H}^{-1}\Phi' - \Psi}{1+w}\right], \qquad
\Omega = 1 + K/\mathcal{H}^2 ,$$
and the frame-invariant variables
$\hat\Delta_i = \Delta_i + \frac{3}{2\beta_2}(1+w_i)\eta$ used in the notes'
*Frame invariant series* subsection.

In [8]:
def restore_k(c, n):
    # reinstate powers of k in the tau^n coefficient c (computed with k=1), using
    # homogeneity: every term is of total degree n in (k, omega)
    c = sympy.cancel(c)
    if c == 0:
        return c
    num, den = sympy.fraction(c)
    md_ = sympy.degree(den, omega) if den.has(omega) else 0
    den0 = sympy.cancel(den / omega**md_)
    assert not den0.has(omega), den
    out = sympy.S.Zero
    for (m,), coef in sympy.Poly(sympy.expand(num), omega).as_dict().items():
        out += coef * omega**m * k ** (n - (m - md_)) / omega**md_ / den0
    return sympy.factor(out)


def eval_form(form, seed_vals):
    # value of a linear form for given seed values (None if it involves an
    # unresolved coefficient beyond the truncation order)
    c = FF.zero
    for key, cf in form.items():
        if key not in seed_vals:
            return None
        c += cf * FF.from_expr(seed_vals[key])
    return c.as_expr()


def mode_series_var(name, seed_vals):
    out = sympy.S.Zero
    for n in range(M + 1):
        sym = coeffs[name][n]
        if sym in fixed_zero:
            continue
        c = (
            seed_vals[sym]
            if sym in seed_vals
            else (None if sym in unresolved or sym not in sol_forms else eval_form(sol_forms[sym], seed_vals))
        )
        if c is None:
            break
        if c != 0:
            out += restore_k(c, n) * t**n
    return out


def expr_forms(expr, nlow=NLOW):
    # Laurent order -> linear form for an auxiliary expression
    lc = laurent_coeffs(expr, nlow=nlow)
    return {n: form_compose(form_from_expr(c), sol_forms) for n, c in lc.items()}


def eval_forms(forms, seed_vals):
    out = sympy.S.Zero
    for n in sorted(forms):
        c = eval_form(forms[n], seed_vals)
        if c is None:
            break
        if c != 0:
            out += restore_k(c, n) * t**n
    return out


zero_seeds = {s: sympy.S.Zero for s in seeds}
modes = {
    "adiabatic": {**zero_seeds, coeffs["eta"][0]: 2 * K_fac},
    "CDM iso": {**zero_seeds, coeffs["clxc"][0]: sympy.S.One},
    "baryon iso": {**zero_seeds, coeffs["clxb"][0]: sympy.S.One},
    "nu density iso": {**zero_seeds, coeffs["clxr"][0]: sympy.S.One},
    "nu velocity iso": {**zero_seeds, coeffs["qr"][0]: sympy.S.One},
}

In [9]:
# auxiliary quantities (linear in the perturbations, background left abstract)
sigma_e = to_fluid_vars(subs(var_subs, sigma))
z_e = to_fluid_vars(subs(var_subs, z))
phi_e = to_fluid_vars(subs(var_subs, phi))
Pi_term = to_fluid_vars(kappa * a**2 * Pi / (2 * k**2))
Phi_e = -phi_e - Pi_term
Psi_e = phi_e - Pi_term
q_e = to_fluid_vars(subs(tot_pert_subs, q))
rho_P_e = to_fluid_vars(subs(tot_subs, rho + P))
P_over_rho = to_fluid_vars(subs(tot_subs, P) / subs(tot_subs, rho))
# comoving-frame eta from the notes' definition in terms of Phi and Psi
etabar_e = 2 * K_fac * (Phi_e + Rational(2, 3) / (1 + K / H**2) * (diff(Phi_e, t) / H - Psi_e) / (1 + P_over_rho))
chi_e = -etabar_e / (2 * K_fac)

aux_exprs = {
    "sigma": sigma_e,
    "z": z_e,
    "phi": phi_e,
    "Pi_t": Pi_term,
    "Phi": Phi_e,
    "Psi": Psi_e,
    "chi": chi_e,
    "hat_clxc": Delta_c + 3 * eta / (2 * K_fac),
    "hat_clxb": Delta_b + 3 * eta / (2 * K_fac),
    "hat_clxg": Delta_g + 2 * eta / K_fac,
    "hat_clxr": Delta_r + 2 * eta / K_fac,
    "vb_sigma": 3 * q_g / 4 + sigma_e,
    "vc_sigma": sigma_e,  # v_c = 0 in the CDM frame
    "qr_sigma": q_r + Rational(4, 3) * sigma_e,
}
aux_forms = {name: expr_forms(e) for name, e in aux_exprs.items()}

results = {}
for label, mode in modes.items():
    results[label] = {name: mode_series_var(name, mode) for name in var_names}
    results[label].update({name: eval_forms(forms, mode) for name, forms in aux_forms.items()})

In [10]:
chi_s, Phi_s, Psi_s = symbols("chi Phi Psi")
aux_lhs = {
    "sigma": sigma,
    "z": z,
    "phi": phi,
    "Pi_t": kappa * a**2 * Pi / (2 * k**2),
    "Phi": Phi_s,
    "Psi": Psi_s,
    "chi": chi_s,
    "hat_clxc": Delta_c + 3 * eta / (2 * K_fac),
    "hat_clxb": Delta_b + 3 * eta / (2 * K_fac),
    "hat_clxg": Delta_g + 2 * eta / K_fac,
    "hat_clxr": Delta_r + 2 * eta / K_fac,
    "vb_sigma": v_b + sigma,
    "vc_sigma": v_c + sigma,
    "qr_sigma": q_r + Rational(4, 3) * sigma,
}


def show_mode(label, names, trunc=Ord):
    # display series truncated to match the orders quoted in the notes: up to tau^trunc,
    # and only the first two nonvanishing orders for the auxiliary quantities
    for name in names:
        orders = sorted(n for n in laurent_dict(results[label][name]) if n <= trunc)
        if name in aux_lhs:
            orders = orders[:2]
        lhs = aux_lhs.get(name, dict(zip(var_names, solve_vars)).get(name))
        dmath(sympy.latex(disp(lhs)) + " = " + series_latex(results[label][name], orders))

### Adiabatic mode ($\chi_0 = -1$)

In [11]:
show_mode("adiabatic", var_names + ["chi", "Psi", "Pi_t", "sigma", "z"])

$\displaystyle \Delta_{g}{\left(\tau \right)} = \frac{\beta_{2} k^{2} \tau^{2}}{3} -  \frac{\beta_{2} k^{2} \omega \tau^{3}}{15}$

$\displaystyle \Delta_{r}{\left(\tau \right)} = \frac{\beta_{2} k^{2} \tau^{2}}{3} -  \frac{\beta_{2} k^{2} \omega \tau^{3}}{15}$

$\displaystyle \Delta_{b}{\left(\tau \right)} = \frac{\beta_{2} k^{2} \tau^{2}}{4} -  \frac{\beta_{2} k^{2} \omega \tau^{3}}{20}$

$\displaystyle \Delta_{c}{\left(\tau \right)} = \frac{\beta_{2} k^{2} \tau^{2}}{4} -  \frac{\beta_{2} k^{2} \omega \tau^{3}}{20}$

$\displaystyle q_{g}{\left(\tau \right)} = \frac{\beta_{2} k^{3} \tau^{3}}{27}$

...

$\displaystyle \chi = -1 + \frac{k^{2} \tau^{2} \left(4 R_{\nu} \beta_{2} + 10 \beta_{2} - 5\right)}{6 \left(4 R_{\nu} + 15\right)}$

$\displaystyle \Psi = - \frac{10}{4 R_{\nu} + 15} -  \frac{25 \omega \tau \left(8 R_{\nu} - 3\right)}{8 \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle \frac{\kappa \Pi{\left(\tau \right)} a^{2}{\left(\tau \right)}}{2 k^{2}} = - \frac{2 R_{\nu}}{4 R_{\nu} + 15} + \frac{35 R_{\nu} \omega \tau}{2 \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle \sigma{\left(\tau \right)} = - \frac{5 k \tau}{4 R_{\nu} + 15} -  \frac{15 k \omega \tau^{2} \left(4 R_{\nu} - 5\right)}{8 \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle z{\left(\tau \right)} = - \frac{\beta_{2} k \tau}{2} + \frac{3 \beta_{2} k \omega \tau^{2}}{20}$

### CDM isocurvature mode

In [12]:
show_mode("CDM iso", var_names + ["Phi", "sigma"])

$\displaystyle \Delta_{g}{\left(\tau \right)} = - \frac{2 R_{c} \omega \tau}{3} + \frac{R_{c} \omega^{2} \tau^{2}}{4} + \frac{R_{c} \omega \tau^{3} \left(28 \beta_{2} k^{2} + 8 k^{2} - 45 \omega^{2}\right)}{540}$

$\displaystyle \Delta_{r}{\left(\tau \right)} = - \frac{2 R_{c} \omega \tau}{3} + \frac{R_{c} \omega^{2} \tau^{2}}{4} + \frac{R_{c} \omega \tau^{3} \left(28 \beta_{2} k^{2} + 8 k^{2} - 45 \omega^{2}\right)}{540}$

$\displaystyle \Delta_{b}{\left(\tau \right)} = - \frac{R_{c} \omega \tau}{2} + \frac{3 R_{c} \omega^{2} \tau^{2}}{16} + \frac{R_{c} \omega \tau^{3} \left(28 \beta_{2} k^{2} + 8 k^{2} - 45 \omega^{2}\right)}{720}$

$\displaystyle \Delta_{c}{\left(\tau \right)} = 1 -  \frac{R_{c} \omega \tau}{2} + \frac{3 R_{c} \omega^{2} \tau^{2}}{16} + \frac{R_{c} \omega \tau^{3} \left(28 \beta_{2} k^{2} - 12 k^{2} - 45 \omega^{2}\right)}{720}$

$\displaystyle q_{g}{\left(\tau \right)} = - \frac{R_{c} k \omega \tau^{2}}{9} + \frac{R_{c} k \omega^{2} \tau^{3} \left(3 R_{c} + R_{\nu} - 4\right)}{36 \left(R_{\nu} - 1\right)}$

...

$\displaystyle G_{3}{\left(\tau \right)} = 0$

$\displaystyle G_{4}{\left(\tau \right)} = 0$

$\displaystyle \eta{\left(\tau \right)} = \frac{R_{c} \beta_{2} \omega \tau}{3} -  \frac{R_{c} \beta_{2} \omega^{2} \tau^{2}}{8} -  \frac{R_{c} \beta_{2} \omega \tau^{3} \left(56 R_{\nu} \beta_{2} k^{2} + 16 R_{\nu} k^{2} - 90 R_{\nu} \omega^{2} + 420 \beta_{2} k^{2} - 330 k^{2} - 675 \omega^{2}\right)}{1080 \left(2 R_{\nu} + 15\right)}$

$\displaystyle \Phi = \frac{R_{c} \omega \tau \left(4 R_{\nu} + 15\right)}{8 \left(2 R_{\nu} + 15\right)} -  \frac{R_{c} \omega^{2} \tau^{2} \left(8 R_{\nu}^{2} + 230 R_{\nu} + 675\right)}{32 \left(2 R_{\nu} + 15\right) \left(2 R_{\nu} + 25\right)}$

$\displaystyle \sigma{\left(\tau \right)} = \frac{R_{c} k \omega \tau^{2} \left(4 R_{\nu} - 15\right)}{24 \left(2 R_{\nu} + 15\right)} -  \frac{R_{c} k \omega^{2} \tau^{3} \left(R_{\nu}^{2} + 35 R_{\nu} - 75\right)}{12 \left(2 R_{\nu} + 15\right) \left(2 R_{\nu} + 25\right)}$

The isocurvature modes take a particularly simple form in terms of the frame-invariant
variables $\hat\Delta_i$ (the perturbations in the frame of vanishing curvature
perturbation $\eta$), reproducing the notes' *Frame invariant series* subsection:

In [13]:
show_mode("CDM iso", ["hat_clxc", "hat_clxb", "hat_clxg", "hat_clxr", "vb_sigma", "vc_sigma", "qr_sigma", "Psi"])

$\displaystyle \Delta_{c}{\left(\tau \right)} + \frac{3 \eta{\left(\tau \right)}}{2 \beta_{2}} = 1 -  \frac{R_{c} k^{2} \omega \tau^{3} \left(4 R_{\nu} - 15\right)}{72 \left(2 R_{\nu} + 15\right)}$

$\displaystyle \Delta_{b}{\left(\tau \right)} + \frac{3 \eta{\left(\tau \right)}}{2 \beta_{2}} = \frac{5 R_{c} k^{2} \omega \tau^{3}}{8 \left(2 R_{\nu} + 15\right)}$

$\displaystyle \Delta_{g}{\left(\tau \right)} + \frac{2 \eta{\left(\tau \right)}}{\beta_{2}} = \frac{5 R_{c} k^{2} \omega \tau^{3}}{6 \left(2 R_{\nu} + 15\right)}$

$\displaystyle \Delta_{r}{\left(\tau \right)} + \frac{2 \eta{\left(\tau \right)}}{\beta_{2}} = \frac{5 R_{c} k^{2} \omega \tau^{3}}{6 \left(2 R_{\nu} + 15\right)}$

$\displaystyle \sigma{\left(\tau \right)} + v_{b}{\left(\tau \right)} = - \frac{15 R_{c} k \omega \tau^{2}}{8 \left(2 R_{\nu} + 15\right)} + \frac{R_{c} k \omega^{2} \tau^{3} \left(4 R_{c} R_{\nu}^{2} + 80 R_{c} R_{\nu} + 375 R_{c} - 24 R_{\nu}^{2} + 165 R_{\nu} - 600\right)}{16 \left(R_{\nu} - 1\right) \left(2 R_{\nu} + 15\right) \left(2 R_{\nu} + 25\right)}$

$\displaystyle \sigma{\left(\tau \right)} + v_{c}{\left(\tau \right)} = \frac{R_{c} k \omega \tau^{2} \left(4 R_{\nu} - 15\right)}{24 \left(2 R_{\nu} + 15\right)} -  \frac{R_{c} k \omega^{2} \tau^{3} \left(R_{\nu}^{2} + 35 R_{\nu} - 75\right)}{12 \left(2 R_{\nu} + 15\right) \left(2 R_{\nu} + 25\right)}$

$\displaystyle q_{r}{\left(\tau \right)} + \frac{4 \sigma{\left(\tau \right)}}{3} = - \frac{5 R_{c} k \omega \tau^{2}}{2 \left(2 R_{\nu} + 15\right)} -  \frac{5 R_{c} k \omega^{2} \tau^{3} \left(4 R_{\nu} - 45\right)}{12 \left(2 R_{\nu} + 15\right) \left(2 R_{\nu} + 25\right)}$

$\displaystyle \Psi = \frac{R_{c} \omega \tau \left(4 R_{\nu} - 15\right)}{8 \left(2 R_{\nu} + 15\right)} -  \frac{R_{c} \omega^{2} \tau^{2} \left(8 R_{\nu}^{2} + 350 R_{\nu} - 675\right)}{32 \left(2 R_{\nu} + 15\right) \left(2 R_{\nu} + 25\right)}$

### Baryon isocurvature mode

There is a null solution with $R_c\Delta_c = -R_b\Delta_b$ constant and everything else
zero (no total density perturbation, so the dynamics is that of the background). The
baryon isocurvature mode is therefore equivalent to a rescaled CDM mode: it equals
$(R_b/R_c)\times$(CDM iso) plus the null mode chosen to make $\Delta_b(0)=1$,
$\Delta_c(0)=0$ — as used in `initial` (`initv(3,:)`).

In [14]:
show_mode("baryon iso", var_names)

# check the exact relation to the CDM isocurvature mode
for name in var_names:
    if name == "clxb":
        ref = (Rb / Rc) * results["CDM iso"]["clxb"] + 1
    elif name == "clxc":
        ref = (Rb / Rc) * (results["CDM iso"]["clxc"] - 1)
    else:
        ref = (Rb / Rc) * results["CDM iso"][name]
    d = sympy.expand(results["baryon iso"][name] - ref)
    assert all(sympy.factor(sympy.cancel(d.coeff(t, n))) == 0 for n in range(Ord + 1)), name
print("baryon iso = (R_b/R_c) x CDM iso + null mode: verified")

$\displaystyle \Delta_{g}{\left(\tau \right)} = \frac{2 \omega \tau \left(R_{c} - 1\right)}{3} -  \frac{\omega^{2} \tau^{2} \left(R_{c} - 1\right)}{4} -  \frac{\omega \tau^{3} \left(R_{c} - 1\right) \left(28 \beta_{2} k^{2} + 8 k^{2} - 45 \omega^{2}\right)}{540}$

$\displaystyle \Delta_{r}{\left(\tau \right)} = \frac{2 \omega \tau \left(R_{c} - 1\right)}{3} -  \frac{\omega^{2} \tau^{2} \left(R_{c} - 1\right)}{4} -  \frac{\omega \tau^{3} \left(R_{c} - 1\right) \left(28 \beta_{2} k^{2} + 8 k^{2} - 45 \omega^{2}\right)}{540}$

$\displaystyle \Delta_{b}{\left(\tau \right)} = 1 + \frac{\omega \tau \left(R_{c} - 1\right)}{2} -  \frac{3 \omega^{2} \tau^{2} \left(R_{c} - 1\right)}{16} -  \frac{\omega \tau^{3} \left(R_{c} - 1\right) \left(28 \beta_{2} k^{2} + 8 k^{2} - 45 \omega^{2}\right)}{720}$

$\displaystyle \Delta_{c}{\left(\tau \right)} = \frac{\omega \tau \left(R_{c} - 1\right)}{2} -  \frac{3 \omega^{2} \tau^{2} \left(R_{c} - 1\right)}{16} -  \frac{\omega \tau^{3} \left(R_{c} - 1\right) \left(28 \beta_{2} k^{2} - 12 k^{2} - 45 \omega^{2}\right)}{720}$

$\displaystyle q_{g}{\left(\tau \right)} = \frac{k \omega \tau^{2} \left(R_{c} - 1\right)}{9} -  \frac{k \omega^{2} \tau^{3} \left(R_{c} - 1\right) \left(3 R_{c} + R_{\nu} - 4\right)}{36 \left(R_{\nu} - 1\right)}$

...

$\displaystyle \pi_{r}{\left(\tau \right)} = \frac{k^{2} \omega \tau^{3} \left(R_{c} - 1\right)}{3 \left(2 R_{\nu} + 15\right)}$

$\displaystyle G_{3}{\left(\tau \right)} = 0$

$\displaystyle G_{4}{\left(\tau \right)} = 0$

$\displaystyle \eta{\left(\tau \right)} = - \frac{\beta_{2} \omega \tau \left(R_{c} - 1\right)}{3} + \frac{\beta_{2} \omega^{2} \tau^{2} \left(R_{c} - 1\right)}{8} + \frac{\beta_{2} \omega \tau^{3} \left(R_{c} - 1\right) \left(56 R_{\nu} \beta_{2} k^{2} + 16 R_{\nu} k^{2} - 90 R_{\nu} \omega^{2} + 420 \beta_{2} k^{2} - 330 k^{2} - 675 \omega^{2}\right)}{1080 \left(2 R_{\nu} + 15\right)}$

baryon iso = (R_b/R_c) x CDM iso + null mode: verified


### Neutrino density isocurvature mode

In [15]:
show_mode("nu density iso", var_names + ["Phi"])

$\displaystyle \Delta_{g}{\left(\tau \right)} = \frac{R_{\nu}}{R_{\nu} - 1} -  \frac{R_{\nu} k^{2} \tau^{2}}{6 \left(R_{\nu} - 1\right)} -  \frac{R_{\nu} k^{2} \omega \tau^{3} \left(R_{c} - 1\right) \left(R_{\nu} - 6\right)}{60 \left(R_{\nu} - 1\right)^{2}}$

$\displaystyle \Delta_{r}{\left(\tau \right)} = 1 -  \frac{k^{2} \tau^{2}}{6} -  \frac{R_{\nu} k^{2} \omega \tau^{3} \left(R_{c} - 1\right)}{60 \left(R_{\nu} - 1\right)}$

$\displaystyle \Delta_{b}{\left(\tau \right)} = - \frac{R_{\nu} k^{2} \tau^{2}}{8 \left(R_{\nu} - 1\right)} -  \frac{R_{\nu} k^{2} \omega \tau^{3} \left(R_{c} - 1\right) \left(R_{\nu} - 6\right)}{80 \left(R_{\nu} - 1\right)^{2}}$

$\displaystyle \Delta_{c}{\left(\tau \right)} = - \frac{R_{\nu} k^{2} \omega \tau^{3} \left(R_{c} - 1\right)}{80 \left(R_{\nu} - 1\right)}$

$\displaystyle q_{g}{\left(\tau \right)} = \frac{R_{\nu} k \tau}{3 \left(R_{\nu} - 1\right)} -  \frac{R_{\nu} k \omega \tau^{2} \left(R_{c} - 1\right)}{4 \left(R_{\nu} - 1\right)^{2}} -  \frac{R_{\nu} k \tau^{3} \left(- 81 R_{c}^{2} \omega^{2} + 27 R_{c} R_{\nu} \omega^{2} + 135 R_{c} \omega^{2} + 8 R_{\nu}^{2} k^{2} - 16 R_{\nu} k^{2} - 27 R_{\nu} \omega^{2} + 8 k^{2} - 54 \omega^{2}\right)}{432 \left(R_{\nu} - 1\right)^{3}}$

...

$\displaystyle \pi_{r}{\left(\tau \right)} = \frac{k^{2} \tau^{2}}{4 R_{\nu} + 15} + \frac{4 R_{\nu} k^{2} \omega \tau^{3}}{3 \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle G_{3}{\left(\tau \right)} = \frac{k^{3} \tau^{3}}{7 \left(4 R_{\nu} + 15\right)}$

$\displaystyle G_{4}{\left(\tau \right)} = 0$

$\displaystyle \eta{\left(\tau \right)} = \frac{R_{\nu} \beta_{2} k^{2} \tau^{2}}{3 \left(4 R_{\nu} + 15\right)} + \frac{R_{\nu} \beta_{2} k^{2} \omega \tau^{3} \left(8 R_{c} R_{\nu}^{2} + 90 R_{c} R_{\nu} + 225 R_{c} - 8 R_{\nu}^{2} - 290 R_{\nu} - 25\right)}{120 \left(R_{\nu} - 1\right) \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle \Phi = - \frac{R_{\nu}}{4 R_{\nu} + 15} -  \frac{R_{\nu} \omega \tau \left(2 R_{\nu} - 15\right)}{4 \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 15\right)}$

### Neutrino velocity isocurvature mode

Note that $\Phi$ and $\Psi$ are singular ($\propto 1/k\tau$) in this mode, whereas the
frame-invariant Weyl potential $\phi$ is regular.

In [16]:
show_mode("nu velocity iso", var_names + ["phi", "sigma", "Phi"])

$\displaystyle \Delta_{g}{\left(\tau \right)} = - \frac{R_{\nu} k \tau}{R_{\nu} - 1} -  \frac{3 R_{\nu} k \omega \tau^{2} \left(R_{c} - 1\right) \left(R_{\nu} - 3\right)}{16 \left(R_{\nu} - 1\right)^{2}} + \frac{R_{\nu} k \tau^{3} \left(108 R_{c}^{2} R_{\nu}^{2} \omega^{2} - 513 R_{c}^{2} R_{\nu} \omega^{2} - 810 R_{c}^{2} \omega^{2} + 180 R_{c} R_{\nu}^{3} \omega^{2} - 171 R_{c} R_{\nu}^{2} \omega^{2} + 801 R_{c} R_{\nu} \omega^{2} + 1620 R_{c} \omega^{2} + 128 R_{\nu}^{3} \beta_{2} k^{2} + 160 R_{\nu}^{3} k^{2} - 180 R_{\nu}^{3} \omega^{2} - 384 R_{\nu}^{2} \beta_{2} k^{2} - 120 R_{\nu}^{2} k^{2} + 63 R_{\nu}^{2} \omega^{2} + 384 R_{\nu} \beta_{2} k^{2} - 240 R_{\nu} k^{2} - 288 R_{\nu} \omega^{2} - 128 \beta_{2} k^{2} + 200 k^{2} - 810 \omega^{2}\right)}{720 \left(R_{\nu} - 1\right)^{3} \left(4 R_{\nu} + 5\right)}$

$\displaystyle \Delta_{r}{\left(\tau \right)} = - k \tau -  \frac{3 R_{\nu} k \omega \tau^{2} \left(R_{c} - 1\right)}{16 \left(R_{\nu} - 1\right)} + \frac{k \tau^{3} \left(27 R_{c}^{2} R_{\nu} \omega^{2} + 45 R_{c} R_{\nu}^{2} \omega^{2} - 99 R_{c} R_{\nu} \omega^{2} + 32 R_{\nu}^{2} \beta_{2} k^{2} + 40 R_{\nu}^{2} k^{2} - 45 R_{\nu}^{2} \omega^{2} - 64 R_{\nu} \beta_{2} k^{2} - 80 R_{\nu} k^{2} + 72 R_{\nu} \omega^{2} + 32 \beta_{2} k^{2} + 40 k^{2}\right)}{720 \left(R_{\nu} - 1\right)^{2}}$

$\displaystyle \Delta_{b}{\left(\tau \right)} = - \frac{3 R_{\nu} k \tau}{4 \left(R_{\nu} - 1\right)} -  \frac{9 R_{\nu} k \omega \tau^{2} \left(R_{c} - 1\right) \left(R_{\nu} - 3\right)}{64 \left(R_{\nu} - 1\right)^{2}} + \frac{R_{\nu} k \tau^{3} \left(108 R_{c}^{2} R_{\nu}^{2} \omega^{2} - 513 R_{c}^{2} R_{\nu} \omega^{2} - 810 R_{c}^{2} \omega^{2} + 180 R_{c} R_{\nu}^{3} \omega^{2} - 171 R_{c} R_{\nu}^{2} \omega^{2} + 801 R_{c} R_{\nu} \omega^{2} + 1620 R_{c} \omega^{2} + 128 R_{\nu}^{3} \beta_{2} k^{2} + 160 R_{\nu}^{3} k^{2} - 180 R_{\nu}^{3} \omega^{2} - 384 R_{\nu}^{2} \beta_{2} k^{2} - 120 R_{\nu}^{2} k^{2} + 63 R_{\nu}^{2} \omega^{2} + 384 R_{\nu} \beta_{2} k^{2} - 240 R_{\nu} k^{2} - 288 R_{\nu} \omega^{2} - 128 \beta_{2} k^{2} + 200 k^{2} - 810 \omega^{2}\right)}{960 \left(R_{\nu} - 1\right)^{3} \left(4 R_{\nu} + 5\right)}$

$\displaystyle \Delta_{c}{\left(\tau \right)} = - \frac{9 R_{\nu} k \omega \tau^{2} \left(R_{c} - 1\right)}{64 \left(R_{\nu} - 1\right)} + \frac{R_{\nu} k \tau^{3} \left(108 R_{c}^{2} R_{\nu} \omega^{2} + 135 R_{c}^{2} \omega^{2} + 180 R_{c} R_{\nu}^{2} \omega^{2} - 171 R_{c} R_{\nu} \omega^{2} - 495 R_{c} \omega^{2} + 128 R_{\nu}^{2} \beta_{2} k^{2} - 180 R_{\nu}^{2} \omega^{2} - 256 R_{\nu} \beta_{2} k^{2} + 63 R_{\nu} \omega^{2} + 128 \beta_{2} k^{2} + 360 \omega^{2}\right)}{960 \left(R_{\nu} - 1\right)^{2} \left(4 R_{\nu} + 5\right)}$

$\displaystyle q_{g}{\left(\tau \right)} = \frac{R_{\nu}}{R_{\nu} - 1} -  \frac{3 R_{\nu} \omega \tau \left(R_{c} - 1\right)}{4 \left(R_{\nu} - 1\right)^{2}} -  \frac{R_{\nu} \tau^{2} \left(- 27 R_{c}^{2} \omega^{2} + 9 R_{c} R_{\nu} \omega^{2} + 45 R_{c} \omega^{2} + 8 R_{\nu}^{2} k^{2} - 16 R_{\nu} k^{2} - 9 R_{\nu} \omega^{2} + 8 k^{2} - 18 \omega^{2}\right)}{48 \left(R_{\nu} - 1\right)^{3}} -  \frac{R_{\nu} \omega \tau^{3} \left(R_{c} - 1\right) \left(81 R_{c}^{2} \omega^{2} - 54 R_{c} R_{\nu} \omega^{2} - 108 R_{c} \omega^{2} + 4 R_{\nu}^{3} k^{2} + 8 R_{\nu}^{2} \beta_{2} k^{2} - 52 R_{\nu}^{2} k^{2} - 16 R_{\nu} \beta_{2} k^{2} + 92 R_{\nu} k^{2} + 54 R_{\nu} \omega^{2} + 8 \beta_{2} k^{2} - 44 k^{2} + 27 \omega^{2}\right)}{192 \left(R_{\nu} - 1\right)^{4}}$

...

$\displaystyle G_{4}{\left(\tau \right)} = \frac{4 k^{3} \tau^{3}}{63 \left(4 R_{\nu} + 5\right)}$

$\displaystyle \eta{\left(\tau \right)} = \frac{2 R_{\nu} \beta_{2} k \tau}{4 R_{\nu} + 5} + \frac{3 R_{\nu} \beta_{2} k \omega \tau^{2} \left(16 R_{c} R_{\nu}^{2} + 80 R_{c} R_{\nu} + 75 R_{c} - 16 R_{\nu}^{2} - 160 R_{\nu} + 5\right)}{32 \left(R_{\nu} - 1\right) \left(4 R_{\nu} + 5\right) \left(4 R_{\nu} + 15\right)} -  \frac{R_{\nu} \beta_{2} k \tau^{3} \left(6048 R_{c}^{2} R_{\nu}^{3} \omega^{2} + 75600 R_{c}^{2} R_{\nu}^{2} \omega^{2} + 255150 R_{c}^{2} R_{\nu} \omega^{2} + 212625 R_{c}^{2} \omega^{2} + 10080 R_{c} R_{\nu}^{4} \omega^{2} + 103824 R_{c} R_{\nu}^{3} \omega^{2} + 148050 R_{c} R_{\nu}^{2} \omega^{2} - 581175 R_{c} R_{\nu} \omega^{2} - 779625 R_{c} \omega^{2} + 7168 R_{\nu}^{4} \beta_{2} k^{2} + 8960 R_{\nu}^{4} k^{2} - 10080 R_{\nu}^{4} \omega^{2} + 143104 R_{\nu}^{3} \beta_{2} k^{2} - 65920 R_{\nu}^{3} k^{2} - 84672 R_{\nu}^{3} \omega^{2} + 181888 R_{\nu}^{2} \beta_{2} k^{2} - 201040 R_{\nu}^{2} k^{2} - 557550 R_{\nu}^{2} \omega^{2} - 821760 R_{\nu} \beta_{2} k^{2} + 564000 R_{\nu} k^{2} + 918225 R_{\nu} \omega^{2} + 489600 \beta_{2} k^{2} - 306000 k^{2} + 283500 \omega^{2}\right)}{10080 \left(R_{\nu} - 1\right)^{2} \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 5\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle \phi{\left(\tau \right)} = \frac{45 R_{\nu} \omega}{4 k \left(4 R_{\nu} + 5\right) \left(4 R_{\nu} + 15\right)} + \frac{15 R_{\nu} \tau \left(256 R_{\nu} \beta_{2} k^{2} - 496 R_{\nu} k^{2} + 84 R_{\nu} \omega^{2} + 960 \beta_{2} k^{2} - 1860 k^{2} - 945 \omega^{2}\right)}{112 k \left(2 R_{\nu} + 15\right) \left(4 R_{\nu} + 5\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle \sigma{\left(\tau \right)} = - \frac{3 R_{\nu}}{4 R_{\nu} + 5} + \frac{45 R_{\nu} \omega \tau}{2 \left(4 R_{\nu} + 5\right) \left(4 R_{\nu} + 15\right)}$

$\displaystyle \Phi = - \frac{3 R_{\nu}}{k \tau \left(4 R_{\nu} + 5\right)} -  \frac{3 R_{\nu} \omega \left(4 R_{\nu} - 15\right)}{4 k \left(4 R_{\nu} + 5\right) \left(4 R_{\nu} + 15\right)}$

## Checks against the notes

We now compare every series quoted in the notes with the derivation above,
coefficient by coefficient (at the orders the notes quote).

In [17]:
x = k * t
b2 = K_fac
Rp15 = 4 * Rv + 15
ot = omega * t

notes = {}
notes["adiabatic"] = {
    "eta": 2 * b2 * (1 - x**2 / 12 * (b2 - 10 / Rp15)),
    "clxg": b2 / 3 * x**2 - b2 / 15 * omega * k**2 * t**3,
    "clxr": b2 / 3 * x**2 - b2 / 15 * omega * k**2 * t**3,
    "clxb": b2 / 4 * x**2 - b2 / 20 * omega * k**2 * t**3,
    "clxc": b2 / 4 * x**2 - b2 / 20 * omega * k**2 * t**3,
    "qg": b2 * x**3 / 27,
    "qr": b2 * x**3 / 27 * (4 * Rv + 23) / Rp15,
    "pir": -Rational(4, 3) * x**2 / Rp15 - omega * k**2 * t**3 / 3 * (4 * Rv - 5) / (Rp15 * (2 * Rv + 15)),
    "G3": -Rational(4, 21) * x**3 / Rp15,
    "chi": -1 + x**2 / 12 * (2 * b2 - 10 * (b2 + 1) / Rp15),
    "Psi": -10 / Rp15 - 25 * ot / 8 * (8 * Rv - 3) / (Rp15 * (2 * Rv + 15)),
    "Pi_t": -2 * Rv / Rp15 + Rational(35, 2) * ot * Rv / ((2 * Rv + 15) * Rp15),
    "sigma": -5 * x / Rp15 - Rational(15, 8) * omega * k * t**2 * (4 * Rv - 5) / (Rp15 * (2 * Rv + 15)),
    "z": -b2 / 2 * x + Rational(3, 20) * b2 * omega * k * t**2,
}
notes["CDM iso"] = {
    "eta": b2 * Rc * ot / 3 - b2 * Rc * ot**2 / 8,
    "clxc": 1 - Rc * ot / 2 + 3 * Rc * ot**2 / 16,
    "clxb": -Rc * ot / 2 + 3 * Rc * ot**2 / 16,
    "clxg": -2 * Rc * ot / 3 + Rc * ot**2 / 4,
    "clxr": -2 * Rc * ot / 3 + Rc * ot**2 / 4,
    "qg": -Rc * omega * k * t**2 / 9,
    "qr": -Rc * omega * k * t**2 / 9,
    "pir": -Rc * omega * k**2 * t**3 / (3 * (2 * Rv + 15)),
    "Phi": Rc * Rp15 * ot / (8 * (2 * Rv + 15)),
    "sigma": Rc * (4 * Rv - 15) * omega * k * t**2 / (24 * (2 * Rv + 15)),
    "hat_clxc": 1 - Rc * (4 * Rv - 15) * omega * k**2 * t**3 / (72 * (2 * Rv + 15)),
    "hat_clxg": Rational(5, 6) * Rc * omega * k**2 * t**3 / (2 * Rv + 15),
    "hat_clxr": Rational(5, 6) * Rc * omega * k**2 * t**3 / (2 * Rv + 15),
    "hat_clxb": Rational(5, 8) * Rc * omega * k**2 * t**3 / (2 * Rv + 15),
    "vb_sigma": -Rational(15, 8) * Rc * omega * k * t**2 / (2 * Rv + 15),
    "vc_sigma": Rc * (4 * Rv - 15) * omega * k * t**2 / (24 * (2 * Rv + 15)),
    "qr_sigma": -Rational(5, 2) * Rc * omega * k * t**2 / (2 * Rv + 15),
    "Psi": Rc * (4 * Rv - 15) * ot / (8 * (2 * Rv + 15)),
}
notes["nu density iso"] = {
    "clxg": -Rv / Rg + Rv / (6 * Rg) * x**2,
    "clxr": 1 - x**2 / 6,
    "clxc": -omega * k**2 * t**3 / 80 * Rv * Rb / Rg,
    "clxb": Rv * x**2 / (8 * Rg),
    "qg": -Rv * x / (3 * Rg) + omega * k * t**2 / 4 * Rv * Rb / Rg**2,
    "qr": x / 3 - x**3 / 54 * (1 + 12 * b2 / Rp15),
    "pir": x**2 / Rp15,
    "eta": b2 * x**2 / 3 * Rv / Rp15,
    "Phi": -Rv / Rp15 - ot / 4 * Rv * (2 * Rv - 15) / (Rp15 * (2 * Rv + 15)),
}
notes["nu velocity iso"] = {
    "clxg": x * Rv / Rg - 3 * omega * k * t**2 / 16 * Rv * Rb * (2 + Rg) / Rg**2,
    "clxr": -x - 3 * omega * k * t**2 * Rv * Rb / (16 * Rg),
    "clxc": -9 * omega * k * t**2 / 64 * Rv * Rb / Rg,
    "clxb": 3 * Rv / (4 * Rg) * x - 9 * omega * k * t**2 / 64 * Rv * Rb * (2 + Rg) / Rg**2,
    "qg": -Rv / Rg
    + 3 * Rv * Rb / (4 * Rg**2) * ot
    + x**2 / 6 * Rv / Rg
    + 3 * ot**2 / 16 * Rv * Rb / Rg**3 * (Rg - 3 * Rb),
    "qr": 1 - x**2 / 6 * (1 + 4 * b2 / (4 * Rv + 5)),
    "pir": 2 * x / (4 * Rv + 5) + omega * k * t**2 * 6 * Rv / ((4 * Rv + 5) * Rp15),
    "G3": Rational(3, 7) * x**2 / (4 * Rv + 5),
    "eta": 2 * b2 * x * Rv / (4 * Rv + 5)
    + omega * k * t**2 * 3 * b2 * Rv / 32 * (Rb / Rg - 80 / ((4 * Rv + 5) * Rp15)),
    "phi": Rational(45, 4) * Rv * omega / (k * Rp15 * (4 * Rv + 5)),
    "sigma": -3 * Rv / (4 * Rv + 5) + 2 * Rational(45, 4) * Rv * omega / (Rp15 * (4 * Rv + 5)) * t,
    "Phi": -3 * Rv / ((4 * Rv + 5) * k * t),
}


def powers_of(expr):
    return sorted(laurent_dict(expr))


mismatches = {}
for label, checks in notes.items():
    for name, notes_expr in checks.items():
        dd = laurent_dict(results[label][name] - notes_expr) if results[label][name] != notes_expr else {}
        diffs = []
        for n in powers_of(notes_expr):
            d = sympy.factor(dd.get(n, 0))
            if d != 0:
                diffs.append((n, d))
        if diffs:
            mismatches[label, name] = diffs
            print(f"MISMATCH  {label:16s} {name:10s} {diffs}")

print(f"\n{sum(len(d) for d in notes.values())} series checked, {len(mismatches)} mismatches")
assert not mismatches


53 series checked, 0 mismatches


## Comparison with the Fortran `initial()` implementation

The same comparison against the expressions coded in `subroutine initial`
(`fortran/equations.f90`), i.e. the `initv` arrays, with ${\tt InitVec} = -{\tt initv(1,:)}$
for the adiabatic mode so that $\chi_0 = -1$. Comparison is at the orders quoted in the
notes (the code carries a few incomplete higher-order factors, such as the
$(1-\omega\tau/5)$ multiplying `initv(1,i_qg)`, which are beyond the accuracy of the
series).

In [18]:
omtau = omega * t
fortran = {}
f_clxg = -b2 / 3 * x**2 * (1 - omtau / 5)
fortran["adiabatic"] = {
    "clxg": f_clxg,
    "clxr": f_clxg,
    "clxb": Rational(3, 4) * f_clxg,
    "clxc": Rational(3, 4) * f_clxg,
    "qg": f_clxg * x / 9,
    "qr": -b2 * (4 * Rv + 23) / Rp15 * x**3 / 27,
    "pir": Rational(4, 3) * x**2 / Rp15 * (1 + omtau / 4 * (4 * Rv - 5) / (2 * Rv + 15)),
    "G3": Rational(4, 21) / Rp15 * x**3,
    "eta": -b2 * 2 * (1 - x**2 / 12 * (-10 / Rp15 + b2)),
}
fortran["adiabatic"] = {key: -v for key, v in fortran["adiabatic"].items()}  # InitVec = -initv
f_clxg = Rc * omtau * (-Rational(2, 3) + omtau / 4)
fortran["CDM iso"] = {
    "clxg": f_clxg,
    "clxr": f_clxg,
    "clxb": f_clxg * Rational(3, 4),
    "clxc": 1 + f_clxg * Rational(3, 4),
    "qg": -Rc / 9 * omtau * x,
    "qr": -Rc / 9 * omtau * x,
    "pir": -Rc * omtau * x**2 / 3 / (2 * Rv + 15),
    "eta": Rc * omtau * (Rational(1, 3) - omtau / 8) * b2,
    "G3": 0,
}
f_iqg = -Rv / Rg * (x / 3 - Rb / 4 / Rg * omtau * x)
fortran["nu density iso"] = {
    "clxg": Rv / Rg * (-1 + x**2 / 6),
    "clxr": 1 - x**2 / 6,
    "clxc": -omtau * x**2 / 80 * Rv * Rb / Rg,
    "clxb": Rv / Rg / 8 * x**2,
    "qg": f_iqg,
    "qr": x / 3,
    "pir": x**2 / Rp15,
    "eta": b2 * Rv / Rp15 / 3 * x**2,
}
f_iqg = Rv / Rg * (-1 + 3 * Rb / 4 / Rg * omtau + x**2 / 6 + 3 * omtau**2 / 16 * Rb / Rg**2 * (Rg - 3 * Rb))
fortran["nu velocity iso"] = {
    "clxg": Rv / Rg * x - 3 * x * omtau / 16 * Rv * Rb * (2 + Rg) / Rg**2,
    "clxr": -x - 3 * x * omtau * Rv * Rb / 16 / Rg,
    "clxc": -9 * omtau * x / 64 * Rv * Rb / Rg,
    "clxb": 3 * Rv / 4 / Rg * x - 9 * omtau * x / 64 * Rv * Rb * (2 + Rg) / Rg**2,
    "qg": f_iqg,
    "qr": 1 - x**2 / 6 * (1 + 4 * b2 / (4 * Rv + 5)),
    "pir": 2 * x / (4 * Rv + 5) + omtau * x * 6 * Rv / Rp15 / (4 * Rv + 5),
    "eta": 2 * b2 * x * Rv / (4 * Rv + 5) + omtau * x * 3 * b2 * Rv / 32 * (Rb / Rg - 80 / Rp15 / (4 * Rv + 5)),
    "G3": Rational(3, 7) * x**2 / (4 * Rv + 5),
}

notes_orders = {label: {name: powers_of(e) for name, e in d.items()} for label, d in notes.items()}
mismatches = {}
for label, checks in fortran.items():
    for name, fexpr in checks.items():
        norders = powers_of(fexpr) if fexpr != 0 else []
        if name in notes_orders[label]:
            norders = [n for n in norders if n in notes_orders[label][name]]
        dd = laurent_dict(results[label][name] - fexpr) if results[label][name] != fexpr else {}
        diffs = []
        for n in norders:
            d = sympy.factor(dd.get(n, 0))
            if d != 0:
                diffs.append((n, d))
        if diffs:
            mismatches[label, name] = diffs
            print(f"MISMATCH  {label:16s} {name:10s} {diffs}")

print(f"\n{sum(len(d) for d in fortran.values())} series checked, {len(mismatches)} mismatches")
assert not mismatches


35 series checked, 0 mismatches


## Cross-check of $\chi$

The same adiabatic $\chi$ series follows from transforming $\eta$ to the comoving frame
using $\bar\eta = \eta - 2\beta_2\mathcal{H} q/[k(\rho+P)]$; the two expressions for
$\bar\eta$ agree through the quoted $(k\tau)^2$ order (differing only at
$O(\omega k^2\tau^3)$). Note that with the $\Phi$, $\Psi$ definition used above the
$O(\omega k^2\tau^3)$ term of $\chi$ vanishes identically.

In [19]:
etabarq_e = eta - 2 * K_fac * H * q_e / rho_P_e / k
chi_q = eval_forms(expr_forms(-etabarq_e / (2 * K_fac)), modes["adiabatic"])
d = laurent_dict(chi_q - results["adiabatic"]["chi"])
assert all(n >= 3 for n in d), d
assert 3 not in laurent_dict(results["adiabatic"]["chi"])
print("chi from the frame transformation of eta agrees through O(k tau)^2")

chi from the frame transformation of eta agrees through O(k tau)^2


The quintessence isocurvature mode quoted in the notes (which requires the scalar field
perturbation equations, with the field velocity strongly damped so that
$\psi \propto \tau^4$ at early times) is not derived here; the other modes are unchanged
by the addition of a quintessence field. The regular vector mode is given in
[astro-ph/0403583](https://arxiv.org/abs/astro-ph/0403583), and the magnetized vector
mode in [astro-ph/0406096](https://arxiv.org/abs/astro-ph/0406096).